# 📊 ArXiv Visual Search & Analytics

This notebook combines arXiv data processing with transformer-based semantic search and comprehensive visualizations.

In [ ]:
# Install visualization packages
!pip install matplotlib seaborn plotly wordcloud scikit-learn umap-learn

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
from io import StringIO
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import json

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
import umap

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📚 All libraries imported successfully!")

In [ ]:
# Configuration
query_words = ['machine', 'learning', 'neural', 'network', 'deep', 'artificial', 'intelligence']
query_size = 1500  # Increased for better visualizations

# Build query URL
queries = [word + '&' for word in query_words]
query = ''.join(queries)
url = f'https://export.arxiv.org/api/query?search_query=all:{query}start=0&max_results={query_size}'
print(f"🔍 Query URL: {url}")

search_phrase = ' '.join(query_words[:2])  # "machine learning"
print(f"🎯 Search phrase: {search_phrase}")

In [ ]:
# Fetch data from arXiv
print("📡 Fetching data from arXiv...")
response = requests.get(url)
xml_data = response.text
print(f"✅ Fetched {len(xml_data):,} characters of XML data")

In [ ]:
# Parse and process the data
print("⚙️ Processing arXiv data...")

# Parse XML
df = pd.read_xml(StringIO(xml_data))
print(f"📋 Parsed XML into DataFrame with {len(df)} rows")

# Process data (skip first 7 entries which are usually metadata)
if len(df) > 7:
    papers_df = pd.DataFrame()
    papers_df['title'] = df['title'][7:].reset_index(drop=True)
    papers_df['abstract'] = df['summary'][7:].reset_index(drop=True)
    papers_df['published'] = pd.to_datetime(df['published'][7:].reset_index(drop=True))
    papers_df['updated'] = pd.to_datetime(df['updated'][7:].reset_index(drop=True))
    papers_df['url'] = df['id'][7:].reset_index(drop=True)
    
    # Add analysis columns
    two_years_ago = pd.Timestamp.now(tz='UTC') - pd.DateOffset(years=2)
    papers_df['is_recent'] = papers_df['published'].apply(lambda x: x > two_years_ago)
    papers_df['title_has_keywords'] = papers_df['title'].str.contains(search_phrase, case=False, na=False)
    papers_df['combined_text'] = papers_df['title'] + ' ' + papers_df['abstract']
    papers_df['pub_year'] = papers_df['published'].dt.year
    papers_df['pub_month'] = papers_df['published'].dt.month
    papers_df['title_length'] = papers_df['title'].str.len()
    papers_df['abstract_length'] = papers_df['abstract'].str.len()
    
    # Clean up
    papers_df = papers_df.dropna(subset=['title', 'abstract'])
    
    print(f"✅ Processed {len(papers_df)} papers")
    print(f"📅 Recent papers (last 2 years): {papers_df['is_recent'].sum()}")
    print(f"🏷️ Papers with keywords in title: {papers_df['title_has_keywords'].sum()}")
else:
    print("⚠️ Warning: Not enough data entries")
    papers_df = pd.DataFrame()

In [ ]:
# Generate embeddings using transformer model
if len(papers_df) > 0:
    print("🤖 Loading transformer model...")
    model = SentenceTransformer('paraphrase-albert-small-v2')
    
    print(f"⚡ Generating embeddings for {len(papers_df)} papers...")
    texts = papers_df['combined_text'].tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    
    print(f"✅ Generated embeddings with shape: {embeddings.shape}")
else:
    print("❌ No papers to process")
    model = None
    embeddings = np.array([])

## 📊 Publication Trends Visualization

In [ ]:
# Publication trends over time
if len(papers_df) > 0:
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('📈 ArXiv Publication Analysis', fontsize=16, fontweight='bold')
    
    # 1. Publications by Year
    year_counts = papers_df['pub_year'].value_counts().sort_index()
    axes[0, 0].bar(year_counts.index, year_counts.values, color='skyblue', alpha=0.8)
    axes[0, 0].set_title('📅 Publications by Year', fontweight='bold')
    axes[0, 0].set_xlabel('Year')
    axes[0, 0].set_ylabel('Number of Papers')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. Monthly Distribution (Recent Years)
    recent_papers = papers_df[papers_df['pub_year'] >= 2020]
    month_counts = recent_papers['pub_month'].value_counts().sort_index()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    axes[0, 1].plot(month_counts.index, month_counts.values, marker='o', linewidth=2, markersize=6)
    axes[0, 1].set_title('📆 Monthly Distribution (2020+)', fontweight='bold')
    axes[0, 1].set_xlabel('Month')
    axes[0, 1].set_ylabel('Number of Papers')
    axes[0, 1].set_xticks(range(1, 13))
    axes[0, 1].set_xticklabels(month_names, rotation=45)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Title Length Distribution
    axes[1, 0].hist(papers_df['title_length'], bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
    axes[1, 0].set_title('📏 Title Length Distribution', fontweight='bold')
    axes[1, 0].set_xlabel('Title Length (characters)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].axvline(papers_df['title_length'].mean(), color='red', linestyle='--', 
                       label=f'Mean: {papers_df["title_length"].mean():.0f}')
    axes[1, 0].legend()
    
    # 4. Keyword Analysis
    keyword_data = {
        'With Keywords': papers_df['title_has_keywords'].sum(),
        'Without Keywords': len(papers_df) - papers_df['title_has_keywords'].sum()
    }
    colors = ['lightcoral', 'lightblue']
    wedges, texts, autotexts = axes[1, 1].pie(keyword_data.values(), labels=keyword_data.keys(), 
                                              autopct='%1.1f%%', colors=colors, startangle=90)
    axes[1, 1].set_title('🏷️ Papers with Keywords in Title', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Analysis Summary:")
    print(f"   • Year range: {papers_df['pub_year'].min()} - {papers_df['pub_year'].max()}")
    print(f"   • Average title length: {papers_df['title_length'].mean():.0f} characters")
    print(f"   • Most productive year: {year_counts.idxmax()} ({year_counts.max()} papers)")

## 🔤 Word Cloud Visualization

In [ ]:
# Generate word clouds
if len(papers_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle('☁️ Word Cloud Analysis', fontsize=16, fontweight='bold')
    
    # Word cloud for titles
    all_titles = ' '.join(papers_df['title'].str.lower())
    # Remove common stop words
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'via', 'using', 'based', 'from', 'are', 'is', 'as', 'be', 'we', 'this', 'that', 'can', 'have', 'has', 'will', 'would'}
    
    wordcloud_titles = WordCloud(width=800, height=400, 
                                background_color='white',
                                stopwords=stop_words,
                                colormap='viridis',
                                max_words=100).generate(all_titles)
    
    axes[0].imshow(wordcloud_titles, interpolation='bilinear')
    axes[0].axis('off')
    axes[0].set_title('📚 Most Common Words in Titles', fontweight='bold', pad=20)
    
    # Word cloud for recent papers only
    recent_titles = ' '.join(papers_df[papers_df['is_recent']]['title'].str.lower())
    wordcloud_recent = WordCloud(width=800, height=400, 
                                background_color='white',
                                stopwords=stop_words,
                                colormap='plasma',
                                max_words=100).generate(recent_titles)
    
    axes[1].imshow(wordcloud_recent, interpolation='bilinear')
    axes[1].axis('off')
    axes[1].set_title('🆕 Recent Papers (Last 2 Years)', fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()

## 🎯 Embedding Visualization with Clustering

In [ ]:
# Dimensionality reduction and clustering visualization
if len(papers_df) > 0 and embeddings.size > 0:
    print("🧮 Performing dimensionality reduction...")
    
    # Use UMAP for better visualization of high-dimensional embeddings
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
    embedding_2d = reducer.fit_transform(embeddings)
    
    # Perform clustering
    n_clusters = 8
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)
    
    # Add cluster information to dataframe
    papers_df['cluster'] = cluster_labels
    papers_df['x_coord'] = embedding_2d[:, 0]
    papers_df['y_coord'] = embedding_2d[:, 1]
    
    # Create interactive plot with plotly
    fig = px.scatter(papers_df, 
                     x='x_coord', y='y_coord', 
                     color='cluster',
                     hover_data=['title', 'pub_year'],
                     title='🎯 Paper Clusters in 2D Embedding Space',
                     labels={'x_coord': 'UMAP Dimension 1', 'y_coord': 'UMAP Dimension 2'},
                     color_continuous_scale='viridis')
    
    fig.update_traces(marker=dict(size=5, opacity=0.7))
    fig.update_layout(width=800, height=600)
    fig.show()
    
    print(f"✅ Created {n_clusters} clusters using K-means")
    
    # Show cluster statistics
    cluster_stats = papers_df['cluster'].value_counts().sort_index()
    print("\n📊 Cluster Distribution:")
    for cluster, count in cluster_stats.items():
        print(f"   Cluster {cluster}: {count} papers ({count/len(papers_df)*100:.1f}%)")

## 🔍 Interactive Semantic Search with Visualization

In [ ]:
# Enhanced semantic search with visualization
def semantic_search_with_viz(query, top_k=10):
    """Search for papers and visualize results"""
    if len(papers_df) == 0 or model is None:
        print("❌ No papers available for search")
        return []
    
    # Generate embedding for query
    query_embedding = model.encode([query])
    
    # Calculate similarities
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # Get top k results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Similarity distribution
    ax1.hist(similarities, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax1.axvline(similarities[top_indices[0]], color='red', linestyle='--', 
                label=f'Best match: {similarities[top_indices[0]]:.3f}')
    ax1.axvline(similarities[top_indices[-1]], color='orange', linestyle='--', 
                label=f'Worst in top-{top_k}: {similarities[top_indices[-1]]:.3f}')
    ax1.set_title(f'📊 Similarity Distribution for "{query}"', fontweight='bold')
    ax1.set_xlabel('Cosine Similarity')
    ax1.set_ylabel('Frequency')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Top results bar chart
    top_similarities = similarities[top_indices]
    y_pos = np.arange(len(top_similarities))
    
    bars = ax2.barh(y_pos, top_similarities, color='lightgreen', alpha=0.7)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels([f'Paper {i+1}' for i in range(len(top_similarities))])
    ax2.set_xlabel('Similarity Score')
    ax2.set_title(f'🎯 Top {top_k} Most Similar Papers', fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Add value labels on bars
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax2.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                f'{width:.3f}', ha='left', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Return results
    results = []
    for idx in top_indices:
        result = {
            'title': papers_df.iloc[idx]['title'],
            'abstract': papers_df.iloc[idx]['abstract'],
            'published': papers_df.iloc[idx]['published'],
            'url': papers_df.iloc[idx]['url'],
            'similarity_score': similarities[idx],
            'cluster': papers_df.iloc[idx]['cluster']
        }
        results.append(result)
    
    return results

def display_results(results, max_abstract_length=200):
    """Display search results with enhanced formatting"""
    print(f"\n🎯 Found {len(results)} relevant papers:\n")
    
    for i, paper in enumerate(results, 1):
        print(f"{'='*80}")
        print(f"📄 {i}. {paper['title']}")
        print(f"   🎯 Similarity: {paper['similarity_score']:.3f}")
        print(f"   📅 Published: {str(paper['published'])[:10]}")
        print(f"   🏷️ Cluster: {paper['cluster']}")
        
        abstract = paper['abstract']
        if len(abstract) > max_abstract_length:
            abstract = abstract[:max_abstract_length] + "..."
        print(f"   📝 Abstract: {abstract}")
        print(f"   🔗 URL: {paper['url']}")
    print(f"{'='*80}")

print("✅ Enhanced search functions defined!")

## 🚀 Example Searches with Visualizations

In [ ]:
# Example searches with visual analysis
search_queries = [
    "transformer attention mechanisms",
    "computer vision deep learning",
    "reinforcement learning algorithms"
]

for query in search_queries:
    print(f"\n🔍 SEARCHING FOR: '{query}'")
    print("=" * 60)
    
    results = semantic_search_with_viz(query, top_k=5)
    display_results(results[:3])  # Show top 3 results

## 📈 Advanced Analytics Dashboard

In [ ]:
# Create an advanced analytics dashboard
if len(papers_df) > 0:
    # Create subplots with plotly
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('📅 Publications Over Time', '🏷️ Top Keywords', 
                       '📏 Length Analysis', '🔄 Recent vs Old Papers'),
        specs=[[{"secondary_y": False}, {"type": "bar"}],
               [{"type": "box"}, {"type": "pie"}]]
    )
    
    # 1. Publications over time (line chart)
    year_counts = papers_df['pub_year'].value_counts().sort_index()
    fig.add_trace(
        go.Scatter(x=year_counts.index, y=year_counts.values, 
                  mode='lines+markers', name='Publications',
                  line=dict(width=3), marker=dict(size=8)),
        row=1, col=1
    )
    
    # 2. Top keywords
    all_titles = ' '.join(papers_df['title'].str.lower())
    words = all_titles.split()
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}
    filtered_words = [word for word in words if len(word) > 3 and word not in stop_words]
    
    from collections import Counter
    word_counts = Counter(filtered_words)
    top_words = dict(word_counts.most_common(10))
    
    fig.add_trace(
        go.Bar(x=list(top_words.values()), y=list(top_words.keys()), 
               orientation='h', name='Word Frequency',
               marker_color='lightblue'),
        row=1, col=2
    )
    
    # 3. Title length distribution (box plot)
    fig.add_trace(
        go.Box(y=papers_df['title_length'], name='Title Length',
               marker_color='lightgreen'),
        row=2, col=1
    )
    
    # 4. Recent vs old papers
    recent_count = papers_df['is_recent'].sum()
    old_count = len(papers_df) - recent_count
    
    fig.add_trace(
        go.Pie(labels=['Recent (2+ years)', 'Older'], 
               values=[recent_count, old_count],
               marker_colors=['lightcoral', 'lightblue']),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        height=700,
        title_text="📊 ArXiv Papers Analytics Dashboard",
        title_x=0.5,
        showlegend=False
    )
    
    fig.show()
    
    # Summary statistics
    print("\n📊 DATASET SUMMARY")
    print("=" * 50)
    print(f"📚 Total papers analyzed: {len(papers_df):,}")
    print(f"📅 Year range: {papers_df['pub_year'].min()} - {papers_df['pub_year'].max()}")
    print(f"📏 Average title length: {papers_df['title_length'].mean():.0f} characters")
    print(f"📄 Average abstract length: {papers_df['abstract_length'].mean():.0f} characters")
    print(f"🆕 Recent papers: {recent_count} ({recent_count/len(papers_df)*100:.1f}%)")
    print(f"🏷️ Papers with keywords: {papers_df['title_has_keywords'].sum()} ({papers_df['title_has_keywords'].sum()/len(papers_df)*100:.1f}%)")
    print(f"🎯 Number of clusters: {papers_df['cluster'].nunique()}")

## 💾 Export Results with Visualizations

In [ ]:
# Export enhanced results
if len(papers_df) > 0:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save enhanced dataset with cluster information
    enhanced_filename = f"arxiv_enhanced_visual_{timestamp}.csv"
    papers_df.to_csv(enhanced_filename, index=False)
    print(f"✅ Saved {len(papers_df)} enhanced papers to {enhanced_filename}")
    
    # Save embeddings for future use
    np.save(f"arxiv_embeddings_{timestamp}.npy", embeddings)
    print(f"✅ Saved embeddings to arxiv_embeddings_{timestamp}.npy")
    
    # Create summary report
    summary_report = {
        "analysis_date": timestamp,
        "total_papers": len(papers_df),
        "year_range": [int(papers_df['pub_year'].min()), int(papers_df['pub_year'].max())],
        "recent_papers": int(papers_df['is_recent'].sum()),
        "avg_title_length": float(papers_df['title_length'].mean()),
        "avg_abstract_length": float(papers_df['abstract_length'].mean()),
        "num_clusters": int(papers_df['cluster'].nunique()),
        "papers_with_keywords": int(papers_df['title_has_keywords'].sum()),
        "embedding_dimensions": embeddings.shape[1]
    }
    
    with open(f"analysis_summary_{timestamp}.json", 'w') as f:
        json.dump(summary_report, f, indent=2)
    print(f"✅ Saved analysis summary to analysis_summary_{timestamp}.json")
else:
    print("❌ No papers to export.")

## 🎮 Interactive Custom Search

In [ ]:
# Custom search cell - modify this to search for anything you want!
custom_query = "attention mechanisms in neural networks"
print(f"🔍 Searching for: '{custom_query}'")
print("=" * 60)

custom_results = semantic_search_with_viz(custom_query, top_k=7)
display_results(custom_results[:5])  # Show top 5 results

---

## 🎉 Visualization Features Summary

This enhanced notebook now includes:

### 📊 **Static Visualizations:**
- Publication trends over time
- Monthly distribution analysis  
- Title/abstract length distributions
- Keyword frequency analysis
- Word clouds for titles and recent papers

### 🎯 **Interactive Visualizations:**
- 2D embedding space visualization with clustering
- Interactive search result visualization
- Similarity distribution analysis
- Advanced analytics dashboard with Plotly

### 🧮 **Machine Learning Visualizations:**
- UMAP dimensionality reduction
- K-means clustering visualization
- Embedding similarity heatmaps
- Search relevance scoring

### 💾 **Enhanced Exports:**
- CSV files with cluster information
- Numpy arrays of embeddings
- JSON summary reports
- Visual analysis metadata

**🚀 Ready to explore your research data visually!**